In [ ]:
# agent 코드에 수합될 task 4코드를 작성해줘
# task 4코드는 Agent.ipynb파일에 포함될 node중하나로 구성될 것이고 아래 평가기준에 맞게끔 재질의(멀티턴)로직이 구현된 FunctionCalling구조야.
# Example ) '최근 많이 오른 지식이 뭐야?'라는 질문이 들어오면, '최근을 정의 하기위한 정확한 날짜를 말해주세요' 라는 답변이 나오고 최종적으로 

# 과제에서 제시한 조건 : 야후파이낸스 API사용 해서 아래 과제 5개에 맞게 답변
# * Task1 - 단순 종목/날짜조회
#    * {종목명}의 {날짜}종가는?
#    * {날짜}에서 거래량 기준 상위 10개 종목은?
# * Task2 - 거래량 조건 검색
#    * 거래량이 전날대비{숫자}% 이상 오른 종목을 모두 보여줘
# * Task3 - 시그널 감지
#    * {날짜} 기준 50일 이동평균을 10%이상 상향 돌파한 종목을 알려줘
# * Task4 - 모호한 의미해석 (ex. 최근 많이 오른 주식, 고점 대비 떨어진 주식 등)
#    * 최근 많이 오른 주식을 알려줘
#    * 고점 대비 많이 떨어진 주식을 알려줘
#    * 평가기준 :
#       * 사람의 모호한 언어를 정량화 하여 표현하는 로직이 고려되었는가
#       * 되묻기(멀티턴) 또는 재질의(재질문) 요구 등 신뢰성있는 답변을 하기 위한 노력이 추가 되었는가

In [ ]:
# 실행해보기
# 아래 셀을 실행하면 KOSPI 데이터와 한국 주식 데이터를 확인할 수 있습니다.

In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
# KRX 티커 파일을 사용한 대규모 다운로드 속도 비교

import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import threading

def load_krx_tickers(market="ALL", sample_size=50):
    """KRX 티커 리스트 로드"""
    try:
        tickers_df = pd.read_csv(
            "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/app/data/krx_tickers_with_market.csv",
            dtype={"code": str}
        )
    except FileNotFoundError:
        try:
            tickers_df = pd.read_csv(
                "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/sasha/test/krx_tickers_cleaned.csv",
                dtype={"code": str}
            )
        except FileNotFoundError:
            print("❌ 종목 리스트 파일을 찾을 수 없습니다.")
            return []
    
    if market.upper() != "ALL":
        tickers_df = tickers_df[tickers_df["market"].str.upper() == market.upper()]
    
    # 샘플링
    if len(tickers_df) > sample_size:
        tickers_df = tickers_df.sample(n=sample_size, random_state=42)
    
    tickers_df["yf_ticker"] = tickers_df.apply(
        lambda x: f"{x['code']}.KS" if x['market'].upper() == "KOSPI" else f"{x['code']}.KQ", 
        axis=1
    )
    
    return tickers_df["yf_ticker"].tolist()

def method1_sequential(tickers, period="1mo"):
    """방법 1: 순차적 다운로드"""
    start_time = time.time()
    success_count = 0
    
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            if not data.empty:
                success_count += 1
        except:
            continue
    
    return time.time() - start_time, success_count

def method2_batch_small(tickers, period="1mo", batch_size=10):
    """방법 2: 소규모 배치 다운로드"""
    start_time = time.time()
    success_count = 0
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            data = yf.download(
                batch,
                period=period,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            if not data.empty:
                success_count += len(batch)
        except:
            continue
    
    return time.time() - start_time, success_count

def method3_batch_large(tickers, period="1mo"):
    """방법 3: 대규모 배치 다운로드"""
    start_time = time.time()
    
    try:
        data = yf.download(
            tickers,
            period=period,
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success_count = len(tickers) if not data.empty else 0
    except:
        success_count = 0
    
    return time.time() - start_time, success_count

def method4_threading(tickers, period="1mo", max_workers=10):
    """방법 4: 멀티스레딩"""
    start_time = time.time()
    success_count = 0
    
    def fetch_single(ticker):
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            return not data.empty
        except:
            return False
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(fetch_single, tickers))
        success_count = sum(results)
    
    return time.time() - start_time, success_count

# 테스트 실행
print("=== KRX 티커를 이용한 대규모 다운로드 속도 비교 ===\n")

# 티커 로드
sample_sizes = [10, 30, 50]

for size in sample_sizes:
    print(f"📊 {size}개 종목 테스트")
    print("-" * 40)
    
    tickers = load_krx_tickers(market="ALL", sample_size=size)
    
    if not tickers:
        print("❌ 티커를 로드할 수 없습니다.\n")
        continue
    
    print(f"로드된 티커 수: {len(tickers)}")
    
    # 방법 1: 순차적
    time1, success1 = method1_sequential(tickers[:10] if size > 10 else tickers)  # 시간 절약을 위해 제한
    print(f"1. 순차적 다운로드: {time1:.2f}초 (성공: {success1}개)")
    
    # 방법 2: 소규모 배치
    time2, success2 = method2_batch_small(tickers, batch_size=5)
    print(f"2. 소규모 배치: {time2:.2f}초 (성공: {success2}개)")
    
    # 방법 3: 대규모 배치
    time3, success3 = method3_batch_large(tickers)
    print(f"3. 대규모 배치: {time3:.2f}초 (성공: {success3}개)")
    
    # 방법 4: 멀티스레딩 (작은 샘플만)
    if size <= 20:
        time4, success4 = method4_threading(tickers, max_workers=5)
        print(f"4. 멀티스레딩: {time4:.2f}초 (성공: {success4}개)")
    
    # 가장 빠른 방법 찾기
    times = [time2, time3]
    methods = ["소규모 배치", "대규모 배치"]
    if size <= 20:
        times.append(time4)
        methods.append("멀티스레딩")
    
    fastest_idx = times.index(min(times))
    print(f"🏆 가장 빠른 방법: {methods[fastest_idx]} ({min(times):.2f}초)")
    print()

print("=== 권장사항 ===")
print("• 10개 이하: 순차적 다운로드")
print("• 10-50개: 소규모 배치 (5-10개씩)")
print("• 50개 이상: 대규모 배치 (주의: 실패 위험)")
print("• 안정성 중시: 멀티스레딩 (max_workers=5-10)")

In [3]:
# KOSPI 지수 심볼: ^KS11
# 개별 한국 주식 심볼 예시: 005930.KS (삼성전자), 000660.KS (SK하이닉스)

period = "1y"  # 데이터 기간 설정 (예: "1d", "5d", "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "ytd", "max")
symbol = "005930.KS"  # 삼성전자 예시

kospi = yf.Ticker(symbol)
data = kospi.history(period=period)

data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-07-29 00:00:00+09:00,80119.155018,80511.895974,79628.228824,79726.414062,12797136,0.0,0.0
2024-07-30 00:00:00+09:00,78940.935417,79530.046875,78548.194444,79530.046875,13169636,0.0,0.0
2024-07-31 00:00:00+09:00,79726.412656,82377.414062,79431.856945,82377.414062,20744323,0.0,0.0
2024-08-01 00:00:00+09:00,84439.301482,84537.486716,81591.929688,81591.929688,20900338,0.0,0.0
2024-08-02 00:00:00+09:00,79530.038572,79922.779503,78057.260080,78155.445312,25800275,0.0,0.0
...,...,...,...,...,...,...,...
2025-07-22 00:00:00+09:00,68100.000000,68500.000000,65600.000000,66000.000000,20829006,0.0,0.0
2025-07-23 00:00:00+09:00,66200.000000,66500.000000,64900.000000,66400.000000,15687058,0.0,0.0
2025-07-24 00:00:00+09:00,66500.000000,66800.000000,65900.000000,66000.000000,12660193,0.0,0.0


In [7]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import asyncio
import concurrent.futures
from typing import List, Tuple, Optional
import threading
import queue

# 방법 1: 스트리밍 방식 - 실시간 결과 출력
def stream_top_performers(tickers: List[str], period_days: int = 30, top_n: int = 10):
    """
    실시간으로 결과를 스트리밍하며 상위 종목 찾기
    """
    results = []
    processed_count = 0
    total_count = len(tickers)
    
    print(f"총 {total_count}개 종목 분석 시작...")
    
    for ticker in tickers:
        try:
            # 개별 종목 빠른 다운로드
            stock = yf.Ticker(ticker)
            hist = stock.history(period=f"{period_days}d", auto_adjust=True)
            
            if len(hist) >= 2:
                start_price = hist['Close'].iloc[0]
                end_price = hist['Close'].iloc[-1]
                return_pct = ((end_price / start_price) - 1) * 100
                
                results.append((ticker, return_pct))
                
                # 실시간 상위 10개 업데이트
                results.sort(key=lambda x: x[1], reverse=True)
                if len(results) > top_n:
                    results = results[:top_n]
                
                processed_count += 1
                
                # 진행상황과 현재 상위 종목 출력
                if processed_count % 50 == 0 or processed_count <= 10:
                    print(f"\n진행: {processed_count}/{total_count}")
                    print("현재 상위 5개:")
                    for i, (sym, ret) in enumerate(results[:5], 1):
                        print(f"  {i}. {sym}: {ret:.2f}%")
            
        except Exception as e:
            continue
    
    return results[:top_n]

# 방법 2: 멀티스레딩 - 병렬 처리
def parallel_top_performers(tickers: List[str], period_days: int = 30, top_n: int = 10, max_workers: int = 20):
    """
    멀티스레딩으로 병렬 처리
    """
    def fetch_single_return(ticker):
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period=f"{period_days}d", auto_adjust=True)
            
            if len(hist) >= 2:
                return_pct = ((hist['Close'].iloc[-1] / hist['Close'].iloc[0]) - 1) * 100
                return (ticker, return_pct)
        except:
            pass
        return None
    
    results = []
    completed = 0
    total = len(tickers)
    
    print(f"병렬 처리 시작 (워커: {max_workers}개)...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 모든 작업 제출
        future_to_ticker = {executor.submit(fetch_single_return, ticker): ticker for ticker in tickers}
        
        # 완료되는 대로 결과 수집
        for future in concurrent.futures.as_completed(future_to_ticker):
            result = future.result()
            if result:
                results.append(result)
            
            completed += 1
            if completed % 100 == 0:
                print(f"진행: {completed}/{total}")
    
    # 상위 종목 정렬
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# 방법 3: 적응형 배치 - 작은 배치로 점진적 처리
def adaptive_batch_processing(tickers: List[str], period_days: int = 30, top_n: int = 10):
    """
    작은 배치로 나누어 점진적 처리
    """
    batch_size = 20  # 작은 배치 크기
    all_results = []
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"배치 {i//batch_size + 1}/{(len(tickers)-1)//batch_size + 1} 처리 중...")
        
        try:
            # 작은 배치 다운로드 (빠름)
            end_date = datetime.now()
            start_date = end_date - timedelta(days=period_days + 10)
            
            data = yf.download(
                batch,
                start=start_date,
                end=end_date,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            
            # 배치 결과 처리
            batch_results = []
            for ticker in batch:
                try:
                    if len(batch) == 1:
                        ticker_data = data
                    else:
                        ticker_data = data[ticker] if ticker in data.columns.get_level_values(0) else None
                    
                    if ticker_data is not None and 'Close' in ticker_data.columns:
                        close_prices = ticker_data['Close'].dropna()
                        if len(close_prices) >= 2:
                            return_pct = ((close_prices.iloc[-1] / close_prices.iloc[0]) - 1) * 100
                            batch_results.append((ticker, return_pct))
                
                except Exception:
                    continue
            
            all_results.extend(batch_results)
            
            # 중간 결과 출력
            all_results.sort(key=lambda x: x[1], reverse=True)
            print(f"현재 상위 3개: {[(t, f'{r:.2f}%') for t, r in all_results[:3]]}")
            
        except Exception as e:
            print(f"배치 {i//batch_size + 1} 처리 실패: {e}")
            continue
    
    return all_results[:top_n]

# 방법 4: 캐시 활용 - 이전 결과 재사용
class CachedAnalyzer:
    def __init__(self):
        self.cache = {}
        self.cache_expiry = {}
    
    def get_top_performers_cached(self, tickers: List[str], period_days: int = 30, top_n: int = 10):
        """캐시를 활용한 분석"""
        current_time = datetime.now()
        cache_duration = timedelta(hours=1)  # 1시간 캐시
        
        fresh_results = []
        tickers_to_fetch = []
        
        # 캐시 확인
        for ticker in tickers:
            cache_key = f"{ticker}_{period_days}"
            if (cache_key in self.cache and 
                cache_key in self.cache_expiry and 
                current_time < self.cache_expiry[cache_key]):
                fresh_results.append(self.cache[cache_key])
            else:
                tickers_to_fetch.append(ticker)
        
        print(f"캐시에서 {len(fresh_results)}개, 새로 가져올 {len(tickers_to_fetch)}개")
        
        # 새 데이터 가져오기 (병렬 처리)
        if tickers_to_fetch:
            new_results = parallel_top_performers(tickers_to_fetch, period_days, len(tickers_to_fetch))
            
            # 캐시 업데이트
            for ticker, return_pct in new_results:
                cache_key = f"{ticker}_{period_days}"
                self.cache[cache_key] = (ticker, return_pct)
                self.cache_expiry[cache_key] = current_time + cache_duration
            
            fresh_results.extend(new_results)
        
        # 최종 정렬
        fresh_results.sort(key=lambda x: x[1], reverse=True)
        return fresh_results[:top_n]

# 사용 예시
if __name__ == "__main__":
    # 테스트용 종목 리스트 (실제로는 더 많은 종목)
    test_tickers = ["AAPL", "GOOGL", "MSFT", "TSLA", "AMZN", "META", "NVDA", "NFLX", "CRM", "ADBE"]
    
    print("=== 방법 1: 스트리밍 방식 ===")
    stream_results = stream_top_performers(test_tickers, 30, 5)
    print("\n=== 방법 1 결과 ===")
    for ticker, return_pct in stream_results:
        print(f"{ticker}: {return_pct:.2f}%")
        
    
    print("\n=== 방법 2: 병렬 처리 ===")
    parallel_results = parallel_top_performers(test_tickers, 30, 5)
    print("\n=== 방법 2 결과 ===")
    for ticker, return_pct in parallel_results:
        print(f"{ticker}: {return_pct:.2f}%")
    
    print("\n=== 방법 3: 적응형 배치 ===")
    adaptive_results = adaptive_batch_processing(test_tickers, 30, 5)
    print("\n=== 방법 3 결과 ===")
    for ticker, return_pct in adaptive_results:
        print(f"{ticker}: {return_pct:.2f}%")
    
    print("\n=== 방법 4: 캐시 활용 ===")
    analyzer = CachedAnalyzer()
    cached_results = analyzer.get_top_performers_cached(test_tickers, 30, 5)
    print("\n=== 방법 4 결과 ===")
    for ticker, return_pct in cached_results:
        print(f"{ticker}: {return_pct:.2f}%")

=== 방법 1: 스트리밍 방식 ===
총 10개 종목 분석 시작...

진행: 1/10
현재 상위 5개:
  1. AAPL: 7.37%

진행: 2/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%

진행: 3/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%
  3. MSFT: 7.28%

진행: 4/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%
  3. MSFT: 7.28%
  4. TSLA: -0.96%

진행: 5/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AMZN: 8.53%
  3. AAPL: 7.37%
  4. MSFT: 7.28%
  5. TSLA: -0.96%

진행: 6/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AMZN: 8.53%
  3. AAPL: 7.37%
  4. MSFT: 7.28%
  5. META: 2.87%

진행: 7/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 8/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 9/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 10/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

=== 방법 1 결과 ===
NVDA: 19.66%
GOOGL: 9.95%
AMZN: 8.53%
AAPL: 7.3

In [8]:
# 각 방법별 시간 측정 및 비교 테스트

import time
from datetime import datetime, timedelta

def time_single_download():
    """개별 다운로드 방식 시간 측정"""
    tickers = ["005930.KS", "000660.KS", "035420.KS","051910.KS", "006400.KS"]

    start_time = time.time()
    results = []

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period="1mo")
            results.append(data)
        except:
            continue

    end_time = time.time()
    return end_time - start_time, len(results)

def time_batch_download():
    """배치 다운로드 방식 시간 측정"""
    tickers = ["005930.KS", "000660.KS", "035420.KS","051910.KS", "006400.KS"]

    start_time = time.time()

    try:
        data = yf.download(
            tickers,
            period="1mo",
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success = not data.empty
    except:
        success = False

    end_time = time.time()
    return end_time - start_time, success

def time_kospi_download():
    """KOSPI 지수 다운로드 시간 측정"""
    start_time = time.time()

    try:
        kospi = yf.Ticker("^KS11")
        data = kospi.history(period="1mo")
        success = not data.empty
    except:
        success = False

    end_time = time.time()
    return end_time - start_time, success

# 시간 측정 실행
print("=== yfinance 다운로드 방식별 시간 비교 ===\n")

# 1. 개별 다운로드
print("1. 개별 다운로드 방식 (5개 종목)")
single_time, single_count = time_single_download()
print(f"   소요시간: {single_time:.2f}초")
print(f"   성공한 종목: {single_count}개")

print()

# 2. 배치 다운로드
print("2. 배치 다운로드 방식 (5개 종목)")
batch_time, batch_success = time_batch_download()
print(f"   소요시간: {batch_time:.2f}초")
print(f"   성공 여부: {'성공' if batch_success else '실패'}")

print()

# 3. KOSPI 지수
print("3. KOSPI 지수 다운로드")
kospi_time, kospi_success = time_kospi_download()
print(f"   소요시간: {kospi_time:.2f}초")
print(f"   성공 여부: {'성공' if kospi_success else '실패'}")

print()

# 결과 비교
print("=== 결과 분석 ===")
if single_count > 0:
    avg_single = single_time / single_count
    print(f"개별 다운로드 - 종목당 평균: {avg_single:.2f}초")

if batch_success and single_count > 0:
    efficiency = single_time / batch_time if batch_time > 0 else 0
    print(f"배치 다운로드 효율성: {efficiency:.1f}배 {'빠름' if efficiency > 1 else '느림'}")

print(f"KOSPI 지수 다운로드가 가장 {'빠름' if kospi_time <= min(single_time, batch_time) else '느림'}")

=== yfinance 다운로드 방식별 시간 비교 ===

1. 개별 다운로드 방식 (5개 종목)
   소요시간: 1.03초
   성공한 종목: 5개

2. 배치 다운로드 방식 (5개 종목)
   소요시간: 1.33초
   성공 여부: 성공

3. KOSPI 지수 다운로드
   소요시간: 0.36초
   성공 여부: 성공

=== 결과 분석 ===
개별 다운로드 - 종목당 평균: 0.21초
배치 다운로드 효율성: 0.8배 느림
KOSPI 지수 다운로드가 가장 빠름


In [9]:
# KRX 티커 파일을 사용한 대규모 다운로드 속도 비교

import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import threading

def load_krx_tickers(market="ALL", sample_size=50):
    """KRX 티커 리스트 로드"""
    try:
        tickers_df = pd.read_csv(
            "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/app/data/krx_tickers_with_market.csv",
            dtype={"code": str}
        )
    except FileNotFoundError:
        try:
            tickers_df = pd.read_csv("/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/sasha/test/krx_tickers_cleaned.csv",
                dtype={"code": str}
            )
        except FileNotFoundError:
            print("❌ 종목 리스트 파일을 찾을 수 없습니다.")
            return []

    if market.upper() != "ALL":
        tickers_df = tickers_df[tickers_df["market"].str.upper() == market.upper()]

    # 샘플링
    if len(tickers_df) > sample_size:
        tickers_df = tickers_df.sample(n=sample_size,random_state=42)

    tickers_df["yf_ticker"] = tickers_df.apply(
        lambda x: f"{x['code']}.KS" if x['market'].upper() == "KOSPI" else f"{x['code']}.KQ",axis=1
    )

    return tickers_df["yf_ticker"].tolist()

def method1_sequential(tickers, period="1mo"):
    """방법 1: 순차적 다운로드"""
    start_time = time.time()
    success_count = 0

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            if not data.empty:
                success_count += 1
        except:
            continue

    return time.time() - start_time, success_count

def method2_batch_small(tickers, period="1mo", batch_size=10):
    """방법 2: 소규모 배치 다운로드"""
    start_time = time.time()
    success_count = 0

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            data = yf.download(
                batch,
                period=period,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            if not data.empty:
                success_count += len(batch)
        except:
            continue

    return time.time() - start_time, success_count

def method3_batch_large(tickers, period="1mo"):
    """방법 3: 대규모 배치 다운로드"""
    start_time = time.time()

    try:
        data = yf.download(
            tickers,
            period=period,
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success_count = len(tickers) if not data.empty else 0
    except:
        success_count = 0

    return time.time() - start_time, success_count

def method4_threading(tickers, period="1mo", max_workers=10):
    """방법 4: 멀티스레딩"""
    start_time = time.time()
    success_count = 0

    def fetch_single(ticker):
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            return not data.empty
        except:
            return False

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(fetch_single,tickers))
        success_count = sum(results)

    return time.time() - start_time, success_count


In [10]:
# 테스트 실행
print("=== KRX 티커를 이용한 대규모 다운로드 속도 비교 ===\n")

# 티커 로드
sample_sizes = [10, 30, 50]

for size in sample_sizes:
    print(f"📊 {size}개 종목 테스트")
    print("-" * 40)

    tickers = load_krx_tickers(market="ALL",
sample_size=size)

    if not tickers:
        print("❌ 티커를 로드할 수 없습니다.\n")
        continue

    print(f"로드된 티커 수: {len(tickers)}")

    # 방법 1: 순차적
    time1, success1 = method1_sequential(tickers[:10] if size > 10 else tickers)  # 시간 절약을 위해 제한
    print(f"1. 순차적 다운로드: {time1:.2f}초 (성공: {success1}개)")

    # 방법 2: 소규모 배치
    time2, success2 = method2_batch_small(tickers, batch_size=5)
    print(f"2. 소규모 배치: {time2:.2f}초 (성공: {success2}개)")

    # 방법 3: 대규모 배치
    time3, success3 = method3_batch_large(tickers)
    print(f"3. 대규모 배치: {time3:.2f}초 (성공: {success3}개)")

    # 방법 4: 멀티스레딩 (작은 샘플만)
    if size <= 20:
        time4, success4 = method4_threading(tickers,max_workers=5)
        print(f"4. 멀티스레딩: {time4:.2f}초 (성공: {success4}개)")

    # 가장 빠른 방법 찾기
    times = [time2, time3]
    methods = ["소규모 배치", "대규모 배치"]
    if size <= 20:
        times.append(time4)
        methods.append("멀티스레딩")

    fastest_idx = times.index(min(times))
    print(f"🏆 가장 빠른 방법: {methods[fastest_idx]} ({min(times):.2f}초)")
    print()

print("=== 권장사항 ===")
print("• 10개 이하: 순차적 다운로드")
print("• 10-50개: 소규모 배치 (5-10개씩)")
print("• 50개 이상: 대규모 배치 (주의: 실패 위험)")
print("• 안정성 중시: 멀티스레딩 (max_workers=5-10)")

=== KRX 티커를 이용한 대규모 다운로드 속도 비교 ===

📊 10개 종목 테스트
----------------------------------------
로드된 티커 수: 10
1. 순차적 다운로드: 2.63초 (성공: 10개)
2. 소규모 배치: 2.66초 (성공: 10개)
3. 대규모 배치: 2.44초 (성공: 10개)
4. 멀티스레딩: 0.67초 (성공: 10개)
🏆 가장 빠른 방법: 멀티스레딩 (0.67초)

📊 30개 종목 테스트
----------------------------------------
로드된 티커 수: 30
1. 순차적 다운로드: 1.19초 (성공: 10개)
2. 소규모 배치: 7.25초 (성공: 30개)
3. 대규모 배치: 6.07초 (성공: 30개)
🏆 가장 빠른 방법: 대규모 배치 (6.07초)

📊 50개 종목 테스트
----------------------------------------
로드된 티커 수: 50
1. 순차적 다운로드: 1.32초 (성공: 10개)
2. 소규모 배치: 14.05초 (성공: 50개)
3. 대규모 배치: 10.41초 (성공: 50개)
🏆 가장 빠른 방법: 대규모 배치 (10.41초)

=== 권장사항 ===
• 10개 이하: 순차적 다운로드
• 10-50개: 소규모 배치 (5-10개씩)
• 50개 이상: 대규모 배치 (주의: 실패 위험)
• 안정성 중시: 멀티스레딩 (max_workers=5-10)


In [ ]:
import redis

# 로컬 호스트, 기본 포트(6379)로 연결
r = redis.Redis(host='localhost', port=6379, db=0)

# 연결 확인 (선택 사항)
try:
    r.ping()
    print("Redis 연결 성공!")
except redis.exceptions.ConnectionError as e:
    print(f"Redis 연결 실패: {e}")
    exit()

Redis 연결 실패: Error 111 connecting to localhost:6379. Connection refused.


: 

In [13]:
import redis
import json
import yfinance as yf
from datetime import datetime

# Redis 연결
redis_client = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

class StockDataManager:
    def __init__(self, redis_client):
        self.redis = redis_client
    
    def save_stock_data(self, ticker, date, stock_data):
        """주식 데이터를 Redis에 저장"""
        key = f"stock:{ticker}:{date}"
        value = {
            "open": float(stock_data['Open']),
            "high": float(stock_data['High']),
            "low": float(stock_data['Low']),
            "close": float(stock_data['Close']),
            "volume": int(stock_data['Volume']),
            "timestamp": date
        }
        self.redis.set(key, json.dumps(value))
        return key
    
    def get_stock_data(self, ticker, date):
        """Redis에서 주식 데이터 조회"""
        key = f"stock:{ticker}:{date}"
        data = self.redis.get(key)
        if data:
            return json.loads(data)
        return None
    
    def get_stock_range(self, ticker, start_date, end_date):
        """특정 기간의 주식 데이터 조회"""
        pattern = f"stock:{ticker}:*"
        keys = self.redis.keys(pattern)
        
        result = {}
        for key in keys:
            date = key.split(':')[2]
            if start_date <= date <= end_date:
                data = self.redis.get(key)
                result[date] = json.loads(data)
        
        return result

# 사용 예시
def example_usage():
    manager = StockDataManager(redis_client)
    
    # yfinance에서 데이터 가져와서 저장
    ticker = "AAPL"
    stock = yf.Ticker(ticker)
    hist = stock.history(period="5d")
    
    for date, row in hist.iterrows():
        date_str = date.strftime('%Y-%m-%d')
        key = manager.save_stock_data(ticker, date_str, row)
        print(f"저장됨: {key}")
    
    # 특정 날짜 데이터 조회
    today = datetime.now().strftime('%Y-%m-%d')
    data = manager.get_stock_data("AAPL", today)
    if data:
        print(f"AAPL {today} 종가: ${data['close']}")
    
    # 기간별 조회
    range_data = manager.get_stock_range("AAPL", "2025-07-24", "2025-07-28")
    for date, info in range_data.items():
        print(f"{date}: 시가 ${info['open']}, 종가 ${info['close']}")

# LangGraph에서 사용할 함수
def get_stock_price_from_redis(ticker: str, date: str = None) -> dict:
    """LangGraph 노드에서 호출할 함수"""
    if not date:
        date = datetime.now().strftime('%Y-%m-%d')
    
    manager = StockDataManager(redis_client)
    return manager.get_stock_data(ticker, date)

if __name__ == "__main__":
    example_usage()

ConnectionError: Error 111 connecting to localhost:6379. Connection refused.

In [12]:
!pip install redis